# [방법론 1 확장 · 2026-10-26] 시작 상태 효과의 일반성 — 관성(α), 시작 시점, 다른 정책 구조

**질문.** 보고 수익을 부풀린 '평가 시작 상태 효과'는 (1) 정책의 관성 때문인가, (2) 시작 시점마다 얼마나 달라지는가,
(3) 구조가 다른 정책(단일 PPO·EIIE·LSRE-CAAN)에도 나타나는가.

**사전 등록(실행 전 커밋).** 학습 없음, 저장된 rt1006 모델만 사용. 미지 구간(2026-07-14~)은 쓰지 않는다
(모든 테스트 구간 끝 = β 정책 테스트 끝 버킷).

| 부분 | 설계 | 판정 기준 |
|---|---|---|
| A 관성 | β 4개 × α ∈ {1, 0.1, 0.03, 0.01} × 시작 {원본(등가중 100%), 현금, 2주 워밍업} × 검증·테스트. 시작 효과 = log 성장(원본) − log 성장(워밍업) | H1: (β, 구간) 8개 계열 중 7개 이상에서 \|효과(α=0.01)\| > \|효과(α=1)\| 이면 '관성이 시작 효과를 키운다' 지지 |
| B 시작 시점 | β=30, α=0.01. 28일 창을 7일 간격으로 이동(검증·테스트 전체). 창마다 원본 시작 − 워밍업 | H2: 창별 시작 효과와 '창 첫 7일 등가중 바스켓 로그 수익'의 피어슨 상관 ≥ 0.7(검증·테스트 합산)이면 '시작 효과는 초기 시장 움직임을 따른다' 지지. 효과의 평균·최소·최대·양수 비율 보고 |
| C 다른 구조 | 단일 PPO(원래 등가중 시작)·EIIE·LSRE-CAAN(원래 현금 시작) × 시작 {현금, 등가중, 워밍업} × 검증(2025-10~12)·테스트 | H3: 원래 시작 기준 효과의 크기가 같은 구간 β=30(α=0.01)의 효과보다 작은 경우가 6개(3정책 × 2구간) 중 5개 이상이면 '관성 없는 구조에선 시작 효과가 작다' 지지 |

- **`런타임 > 모두 실행`만 누르면 된다.** CPU로 충분, 약 1~2시간. 학습 없음(베이스라인은 저장 모델을 불러옴).
- 드라이브 연결 창에서 **ksbdaniel7@gmail.com** 계정을 고른다.
- 결과: `data/crypto/rt1006/start_state_alpha.csv`, `start_state_rolling.csv`, `start_state_baselines.csv`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정)을 불러와 셀을 순서대로 실행한다.
# (1) 설정 셀 직후 모델·결과 경로를 새 폴더(rt1006)로 바꿔 기존 결과를 덮어쓰지 않는다.
# (2) 모든 .learn() 호출 결과를 캐시에 저장 — 끊겨도 `모두 실행`을 다시 누르면 학습된 부분은 건너뛴다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = 'rt1006'
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True); CRYPTO_DIR.mkdir(parents=True, exist_ok=True)
POOL_DIR = MODEL_DIR / 'crypto_hrl_pool'
ORIG_PPO_PATH = MODEL_DIR / 'crypto_ppo_portfolio.zip'
for _n in ('EIIE_MODEL_DIR', 'LSRE_CAAN_MODEL_DIR'):
    if _n in globals():
        globals()[_n] = MODEL_DIR / Path(globals()[_n]).name
        globals()[_n].mkdir(parents=True, exist_ok=True)
CACHE = MODEL_DIR / '_learn_cache'; CACHE.mkdir(exist_ok=True)
import stable_baselines3 as _sb3
_BASE_PPO, _BASE_DQN = _sb3.PPO, _sb3.DQN
def _cached(base):
    class _C(base):
        def learn(self, total_timesteps, *a, **k):
            _CTR[0] += 1
            p = CACHE / f'{_NB[0]}_c{_CUR[0]}_{_CTR[0]}_{base.__name__}.zip'
            if p.exists():
                self.set_parameters(str(p), exact_match=True, device=self.device)
                self.num_timesteps += total_timesteps
                print(f'[캐시] {p.name} 로드 — 학습 생략')
                return self
            out = super().learn(total_timesteps, *a, **k)
            self.save(str(p))
            return out
    _C.__name__ = _C.__qualname__ = base.__name__
    return _C
PPO, DQN = _cached(_BASE_PPO), _cached(_BASE_DQN)
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

_NB, _CUR, _CTR = [''], [0], [0]

def run_cells(path, ids, patch_after, extra=None, subs=None):
    """extra: 경로 변경 직후 실행할 코드(예: β 변경). subs: {셀번호: [(원문, 대체), ...]} 문자열 치환."""
    cells = fetch_cells(path)
    _NB[0] = Path(path).stem
    for i in ids:
        _CUR[0], _CTR[0] = i, 0
        src = cells[i]
        for old, new in (subs or {}).get(i, []):
            assert old in src, f'셀 {i}에 치환 대상 없음: {old!r}'
            src = src.replace(old, new)
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        r = _ip.run_cell(src)
        if not r.success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after:
            for code in (PATCH, extra):
                if code and not _ip.run_cell(code).success:
                    raise RuntimeError('경로/설정 변경 실패')
print('실행기 준비 완료')


In [ ]:
# 준비: 데이터·환경·평가 함수 (β=30으로 치환) — 원본 crypto_beta_robustness_colab(커밋 0e86a41) 셀 재사용
run_cells('crypto/notebooks/crypto_beta_robustness_colab.ipynb', [3, 5, 6, 8, 10, 12, 13, 15, 17], patch_after=3, extra="BETA = 30   # 재학습 후 검증 샤프 최고 β\nPOOL_DIR_ROBUST = MODEL_DIR / 'crypto_beta_robustness'\nDEPLOYED_AGENT_PATH = MODEL_DIR / 'crypto_hrl_pool_v4' / f'agent_beta_{BETA}.zip'\nassert DEPLOYED_AGENT_PATH.exists(), f'재학습한 v4 β={BETA} 모델 없음: {DEPLOYED_AGENT_PATH}'\nprint(f'[β 변경] BETA={BETA} | 기준 모델: {DEPLOYED_AGENT_PATH}')\n")


In [ ]:
%%time
# A. 관성(α)과 시작 상태 — β 정책
import numpy as np, pandas as pd
WARMUP = 14 * 48
BETAS = [-90, -30, 30, 90]
ALPHA_GRID = [1.0, 0.1, 0.03, 0.01]
POOL_V4 = MODEL_DIR / 'crypto_hrl_pool_v4'
models = {b: PPO.load(str(POOL_V4 / f'agent_beta_{b}.zip')) for b in BETAS}
TEST_END_BUCKET = test_buckets[-1]          # 이후 모든 테스트 구간의 끝(미지 구간 미사용)
SPLITS = {'valid': list(valid_buckets), 'test': list(test_buckets)}


def beta_run(model, buckets, start, alpha):
    # attribution 노트북 policy_path와 같은 루프. start: 'orig'(env 기본 = 등가중 100%) / 'cash' / 'warmup'
    if start == 'warmup':
        i0 = all_buckets.index(buckets[0]); lo = max(0, i0 - WARMUP)
        run_bk, skip = all_buckets[lo:i0] + list(buckets), i0 - lo
    else:
        run_bk, skip = list(buckets), 0
    env = SparseGatedCryptoPortfolioEnv(buckets=run_bk, top_k=TOP_K, **env_kwargs)
    obs, _ = env.reset()
    if start in ('warmup', 'cash'):
        env.weights = np.zeros(env.n_assets, dtype=np.float32); obs = env._get_obs()
    done = False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        tw, rho, top_idx = env.action_to_weights(a)
        w = (1 - alpha) * env.weights + alpha * tw
        obs, _, done, _, _ = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
    v = np.asarray(env.value_history, dtype=np.float64)
    return float(np.log(v[-1] / v[skip]))


# 재현 확인: 이전 분해 노트북과 같은 값이 나와야 한다(β=30, α=0.01)
_chk = {('valid', 'orig'): 0.029832944702193, ('valid', 'warmup'): -0.0008248758856163052,
        ('test', 'orig'): 0.08729419859314569, ('test', 'warmup'): 0.028605417897803687}
for (sp, st), ret in _chk.items():
    g = beta_run(models[30], SPLITS[sp], st, 0.01)
    assert abs(np.expm1(g) - ret) < 1e-6, (sp, st, np.expm1(g), ret)
print('재현 확인 통과')

rows = []
for sp, bk in SPLITS.items():
    for beta, m in models.items():
        for alpha in ALPHA_GRID:
            for st in ('orig', 'cash', 'warmup'):
                rows.append({'구간': sp, 'β': beta, 'α': alpha, '시작': st, 'log 성장': beta_run(m, bk, st, alpha)})
            r = rows[-3:]
            print(f"{sp} β={beta} α={alpha}: 원본 {r[0]['log 성장']*100:+.2f} 현금 {r[1]['log 성장']*100:+.2f} 워밍업 {r[2]['log 성장']*100:+.2f} 로그%p", flush=True)
alpha_df = pd.DataFrame(rows)
alpha_df.to_csv(CRYPTO_DIR / 'start_state_alpha.csv', index=False, encoding='utf-8-sig')
print('저장: start_state_alpha.csv')


In [ ]:
%%time
# B. 시작 시점 — β=30, α=0.01, 28일 창 7일 간격
WIN, STEP = 28 * 48, 7 * 48
_probe = SparseGatedCryptoPortfolioEnv(buckets=SPLITS['test'][:2], top_k=TOP_K, **env_kwargs)


def ew_log_return(bks):
    g = 0.0
    for b in bks:
        r = np.asarray(_probe._asset_ret(b), dtype=np.float64); t = np.asarray(_probe._tradable(b), dtype=bool)
        g += np.log1p(r[t].mean() if t.any() else 0.0)
    return g


roll = []
for sp, bk in SPLITS.items():
    for s in range(0, len(bk) - WIN + 1, STEP):
        sub = bk[s:s + WIN]
        g_o, g_w = beta_run(models[30], sub, 'orig', 0.01), beta_run(models[30], sub, 'warmup', 0.01)
        roll.append({'구간': sp, '창 시작': str(bucket_to_ts(sub[0])), '원본': g_o, '워밍업': g_w, '시작 효과': g_o - g_w,
                     '첫 7일 등가중 log': ew_log_return(sub[:7 * 48])})
        print(f"{sp} {roll[-1]['창 시작'][:10]}: 시작 효과 {roll[-1]['시작 효과']*100:+.2f} | 첫 7일 등가중 {roll[-1]['첫 7일 등가중 log']*100:+.2f} 로그%p", flush=True)
roll_df = pd.DataFrame(roll)
roll_df.to_csv(CRYPTO_DIR / 'start_state_rolling.csv', index=False, encoding='utf-8-sig')
print('저장: start_state_rolling.csv')


In [ ]:
# C. 다른 구조 — 공통 실행 함수 (정책 재학습 없음)
BASE_ROWS = []


def base_run(make_env, model, buckets, allb, start):
    # start: 'cash' / 'ew' / 'warmup'(env 기본 시작에서 2주 먼저 돌린 뒤 평가)
    if start == 'warmup':
        i0 = allb.index(buckets[0]); lo = max(0, i0 - WARMUP)
        run_bk, skip = allb[lo:i0] + list(buckets), i0 - lo
    else:
        run_bk, skip = list(buckets), 0
    env = make_env(run_bk)
    obs, _ = env.reset()
    n, L = env.n_assets, len(env.weights)
    if start == 'cash':
        w = np.zeros(L, dtype=np.float32)
        if L > n: w[-1] = 1.0
        env.weights = w; obs = env._get_obs()
    elif start == 'ew':
        t = np.asarray(env._tradable(run_bk[0]), dtype=np.float32)[:n]
        w = np.zeros(L, dtype=np.float32); w[:n] = t / max(t.sum(), 1.0)
        env.weights = w; obs = env._get_obs()
    vals, done = [float(env.portfolio_value)], False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, _ = env.step(a)
        vals.append(float(env.portfolio_value))
    return float(np.log(vals[-1] / vals[skip]))


def eval_baseline(name, orig_start, make_env, model, allb, valid_bk, test_bk):
    for sp, bk in (('valid', valid_bk), ('test', test_bk)):
        for st in ('cash', 'ew', 'warmup'):
            BASE_ROWS.append({'정책': name, '원래 시작': orig_start, '구간': sp, '시작': st,
                              'log 성장': base_run(make_env, model, bk, allb, st)})
        r = BASE_ROWS[-3:]
        print(f"{name} {sp}: 현금 {r[0]['log 성장']*100:+.2f} 등가중 {r[1]['log 성장']*100:+.2f} 워밍업 {r[2]['log 성장']*100:+.2f} 로그%p", flush=True)
    pd.DataFrame(BASE_ROWS).to_csv(CRYPTO_DIR / 'start_state_baselines.csv', index=False, encoding='utf-8-sig')


_V0, _V1 = '2025-10-01', '2025-12-31'
print('준비 완료')


In [ ]:
%%time
run_cells('crypto/notebooks/crypto_eiie_baseline_colab.ipynb', [3, 5, 7, 9, 11, 13, 15, 18], patch_after=3)
_vb = [b for b in all_buckets if _V0 <= bucket_to_date(b) <= _V1]
_tb = [b for b in test_buckets if b <= TEST_END_BUCKET]
eval_baseline('EIIE', 'cash', lambda bk: EIIEPortfolioEnv(bk, symbols, close_mat, high_mat, low_mat, pos_of),
              eiie_model, list(all_buckets), _vb, _tb)


In [ ]:
%%time
run_cells('crypto/notebooks/crypto_lsre_caan_baseline_colab.ipynb', [3, 5, 7, 9, 11, 13, 15, 18], patch_after=3)
_vb = [b for b in all_buckets if _V0 <= bucket_to_date(b) <= _V1]
_tb = [b for b in test_buckets if b <= TEST_END_BUCKET]
eval_baseline('LSRE-CAAN', 'cash', lambda bk: PriceWindowPortfolioEnv(bk, symbols, close_mat, high_mat, low_mat, pos_of),
              lsre_caan_model, list(all_buckets), _vb, _tb)


In [ ]:
%%time
run_cells('crypto/notebooks/crypto_rl_trading_colab.ipynb', [2, 4, 6, 8, 9], patch_after=2)
single_ppo = PPO.load(str(ORIG_PPO_PATH))
_vb = [b for b in all_buckets if _V0 <= bucket_to_date(b) <= _V1]
_tb = [b for b in test_buckets if b <= TEST_END_BUCKET]
eval_baseline('단일 PPO', 'ew', lambda bk: CryptoPortfolioEnv(bk, **env_kwargs), single_ppo, list(all_buckets), _vb, _tb)


In [ ]:
# 사전 등록 판정
A = alpha_df.pivot_table(index=['구간', 'β', 'α'], columns='시작', values='log 성장')
A['효과'] = A['orig'] - A['warmup']
h1 = [(sp, b, abs(A.loc[(sp, b, 0.01), '효과']) > abs(A.loc[(sp, b, 1.0), '효과'])) for sp in SPLITS for b in BETAS]
n1 = sum(x[2] for x in h1)
print((A['효과'] * 100).unstack('α').round(2).to_string())
print(f"\n[H1] |효과(α=0.01)| > |효과(α=1)|: {n1}/8 → {'지지' if n1 >= 7 else '지지 안 됨'}")

r = np.corrcoef(roll_df['시작 효과'], roll_df['첫 7일 등가중 log'])[0, 1]
e = roll_df['시작 효과'] * 100
print(f"\n[H2] 창 {len(roll_df)}개, 상관 {r:+.3f} → {'지지' if r >= 0.7 else '지지 안 됨'} | 효과 평균 {e.mean():+.2f}, 최소 {e.min():+.2f}, 최대 {e.max():+.2f} 로그%p, 양수 {np.mean(e > 0):.0%}")

B = pd.DataFrame(BASE_ROWS)
n3, tot = 0, 0
for (name, orig), g in B.groupby(['정책', '원래 시작'], sort=False):
    for sp in SPLITS:
        gg = g[g['구간'] == sp].set_index('시작')['log 성장']
        eff = gg[orig] - gg['warmup']
        ref = A.loc[(sp, 30, 0.01), '효과']
        n3 += abs(eff) < abs(ref); tot += 1
        print(f"{name} {sp}: 원래 시작({orig}) 효과 {eff*100:+.2f} vs β=30 {ref*100:+.2f} 로그%p")
print(f"\n[H3] 베이스라인 효과가 β=30보다 작음: {n3}/{tot} → {'지지' if n3 >= 5 else '지지 안 됨'}")
